# Random Forest Flood Mapping

2-stage RF training pipeline for flood detection using STURM-Flood SAR tiles.

## Design

All model selection (pct + depth search) happens **within STURM-Flood only**,
using an 80/20 event-level split for train/validation.
External EMSR test areas are only used after the final model is chosen.

## Workflow

| Cell | Step | Description |
|------|------|-------------|
| 1    | Config | Parameters, folder pickers |
| 1b   | Copy  | Copy data to local SSD |
| 2    | Imports | Libraries |
| 3    | Verify | Check data folders |
| 4    | Helpers | Load/eval functions |
| 5    | **Event split** | 80/20 on ems_code, fixed seed |
| 6    | **Pct test** | Learning curve on val split |
| 7    | **Depth test** | max_depth search on val split |
| 8    | **Final model** | Train + save .joblib + .json |
| 9    | **External test** | Run on all EMSR areas, save metrics + GeoTIFF |
| 10   | **Re-eval (standalone)** | Re-run inference on a single area without retraining |

**Setup:** Cell 1 to Cell 1b to Cell 2 to Cell 3 to Cell 4 to Cell 5


## Cell 1: Configuration

Set all experiment parameters and select data folders.

**Key parameters to update between steps:**
- After pct test: update `best_train_pct`
- After depth test: update `best_max_depth`


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import os, glob

# --- MOUNT GOOGLE DRIVE ---
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# OUTPUT FOLDER STRUCTURE
# ============================================================
output_base = '/content/drive/MyDrive/Examensarbete/RF_results_v16'
dirs = {
    'models':  os.path.join(output_base, 'models'),
    'configs': os.path.join(output_base, 'configs'),
    'tables':  os.path.join(output_base, 'tables'),
    'geotiff': os.path.join(output_base, 'geotiff_predictions'),
}
for d in dirs.values():
    os.makedirs(d, exist_ok=True)
print('Output folders ready:')
for k, v in dirs.items():
    print(f'  {k}: {v}')

# ============================================================
# PATH TO STURM-FLOOD METADATA CSV
# ============================================================
metadata_path = '/content/drive/MyDrive/Examensarbete/Dataset_STURMFLOOD/Sentinel1_metadata_fixed.csv'

# ============================================================
# SCAN DRIVE FOR FOLDERS WITH .TIF FILES
# ============================================================
def find_tif_folders(folder_path, max_depth=4):
    tif_folders = []
    for root, dirs_, files_list in os.walk(folder_path):
        depth = root.replace(folder_path, '').count(os.sep)
        if depth < max_depth:
            tif_count = sum(1 for f in files_list if f.lower().endswith(('.tif', '.tiff')))
            if tif_count > 0:
                rel_path = root.replace('/content/drive/MyDrive/', '')
                tif_folders.append((f"{rel_path}  ({tif_count} tiles)", root))
    return sorted(tif_folders)

print("\nScanning Google Drive for folders with .tif files...")
tif_folders = find_tif_folders('/content/drive/MyDrive/')
print(f"Found {len(tif_folders)} folders\n")

def make_folder_picker(label):
    dropdown = widgets.Dropdown(
        options=[(name, path) for name, path in tif_folders],
        description=label,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='80%')
    )
    button = widgets.Button(description="Select this folder",
                            button_style='primary', icon='check')
    output = widgets.Output()
    selected = {'path': None}
    def on_button_click(b):
        with output:
            clear_output()
            selected['path'] = dropdown.value
            n_tifs = len(glob.glob(os.path.join(dropdown.value, '*.tif')))
            print(f"Selected: {dropdown.value}")
            print(f"   {n_tifs} .tif files")
    button.on_click(on_button_click)
    display(widgets.HTML(f'<b>{label}</b>'), dropdown, button, output)
    return selected

# STURM-Flood training data
sturm_s1_sel   = make_folder_picker('STURM-Flood S1 tiles (training radar):')
print()
sturm_mask_sel = make_folder_picker('STURM-Flood masks (training labels):')
print()

# External EMSR test areas
print('─── EXTERNAL TEST AREAS ──────────────────────')
karlstad_s1_sel   = make_folder_picker('Karlstad S1 tiles (EMSR427):')
print()
karlstad_mask_sel = make_folder_picker('Karlstad flood masks (EMSR427):')
print()
goteborg_s1_sel   = make_folder_picker('Göteborg S1 tiles (EMSR427):')
print()
goteborg_mask_sel = make_folder_picker('Göteborg flood masks (EMSR427):')
print()
malung_s1_sel     = make_folder_picker('Malung S1 tiles (EMSR280):')
print()
malung_mask_sel   = make_folder_picker('Malung flood masks (EMSR280):')

# ============================================================
# EVENT SPLIT
# ============================================================
split_seed  = 42     # seed for reproducible 80/20 event split
train_ratio = 0.80   # fraction of events for training

# ============================================================
# PCT TEST  (Cell 6)
# ============================================================
# Percentage of each event's tiles to use, as integers (e.g. 10 = 10%)
train_pct_list = [25, 30]
val_pct        = 50       # % of val-event tiles to load for evaluation
fixed_depth    = 10      # max_depth held constant during pct test

# ============================================================
# DEPTH TEST  (Cell 7)
# ============================================================
# Update best_train_pct after running Cell 6
best_train_pct = 5      # %, set to best value from pct test
depth_list     = [6, 8, 10, 12, 14, 16, None]

# ============================================================
# FINAL MODEL  (Cell 8)
# ============================================================
# Update best_max_depth after running Cell 7
best_max_depth = 10

# ============================================================
# RF HYPERPARAMETERS (shared across all steps)
# ============================================================
rf_n_estimators = 100
rf_max_features = 'sqrt'
rf_class_weight = 'balanced'   # handles class imbalance; no pixel subsampling
rf_n_jobs       = -1
score_threshold = 0.5
tile_seed       = 42    # seed for tile shuffling within events
rf_seed         = 42    # seed for RandomForestClassifier


## Cell 1b: Copy data to local Colab disk

Run after selecting all folders in Cell 1.
Copying from Drive to local SSD reduces file-read latency 10–50x.


In [ ]:
import shutil
from tqdm import tqdm

def copytree_progress(src, dst, label):
    os.makedirs(dst, exist_ok=True)
    files = [f for f in os.listdir(src) if f.endswith('.tif')]
    for f in tqdm(files, desc=label):
        shutil.copy2(os.path.join(src, f), os.path.join(dst, f))

print("Copying data to local Colab disk...")
copytree_progress(sturm_s1_sel['path'],       '/content/sturm_s1/',       'STURM S1')
copytree_progress(sturm_mask_sel['path'],     '/content/sturm_mask/',     'STURM masks')
copytree_progress(karlstad_s1_sel['path'],    '/content/karlstad_s1/',    'Karlstad S1')
copytree_progress(karlstad_mask_sel['path'],  '/content/karlstad_mask/',  'Karlstad masks')
copytree_progress(goteborg_s1_sel['path'],    '/content/goteborg_s1/',    'Göteborg S1')
copytree_progress(goteborg_mask_sel['path'],  '/content/goteborg_mask/',  'Göteborg masks')
copytree_progress(malung_s1_sel['path'],      '/content/malung_s1/',      'Malung S1')
copytree_progress(malung_mask_sel['path'],    '/content/malung_mask/',    'Malung masks')
print("\nAll data copied to local disk.")

# ── Local paths ───────────────────────────────────────────────────────────────
sturm_s1_dir   = '/content/sturm_s1/'
sturm_mask_dir = '/content/sturm_mask/'

# External test areas dict, used in Cell 9
emsr_areas = {
    'karlstad': {
        's1_dir':   '/content/karlstad_s1/',
        'mask_dir': '/content/karlstad_mask/',
    },
    'goteborg': {
        's1_dir':   '/content/goteborg_s1/',
        'mask_dir': '/content/goteborg_mask/',
    },
    'malung': {
        's1_dir':   '/content/malung_s1/',
        'mask_dir': '/content/malung_mask/',
    },
}


## Cell 2: Imports


In [ ]:
import os, glob, time, json, csv
from datetime import datetime

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
import joblib
import gc

print('Imports OK.')


## Cell 3: Verify data

Check that all configured folders exist and contain tiles.


In [ ]:
checks = [
    ('STURM S1',       sturm_s1_dir),
    ('STURM masks',    sturm_mask_dir),
    ('Karlstad S1',    emsr_areas['karlstad']['s1_dir']),
    ('Karlstad masks', emsr_areas['karlstad']['mask_dir']),
    ('Göteborg S1',    emsr_areas['goteborg']['s1_dir']),
    ('Göteborg masks', emsr_areas['goteborg']['mask_dir']),
    ('Malung S1',      emsr_areas['malung']['s1_dir']),
    ('Malung masks',   emsr_areas['malung']['mask_dir']),
]
for name, path in checks:
    if not os.path.isdir(path):
        print(f'ERROR  {name}: not found -> {path}')
    else:
        n = len(glob.glob(os.path.join(path, '*.tif')))
        print(f'{"OK   " if n > 0 else "WARN "}  {name}: {n} tiles')

# Build file lists for EMSR areas
for area, cfg in emsr_areas.items():
    s1_files   = sorted(glob.glob(os.path.join(cfg['s1_dir'],   '*.tif')))
    mask_files = sorted(glob.glob(os.path.join(cfg['mask_dir'], '*.tif')))
    cfg['s1_files']    = s1_files
    cfg['mask_lookup'] = {os.path.basename(f): f for f in mask_files}
    print(f'  {area}: {len(s1_files)} S1 tiles, {len(mask_files)} masks')


## Cell 4: Helper functions

Load tiles, evaluate model, append CSV rows.


In [ ]:
def load_tile_pixels(s1_path, mask_path):
    """Load all valid pixels from one S1 tile + mask. Returns X (n,2), y (n,)."""
    with rasterio.open(s1_path) as src:
        img = src.read()
    with rasterio.open(mask_path) as src:
        mask = src.read(1)
    X         = np.column_stack([img[0].flatten(), img[1].flatten()])
    mask_flat = mask.flatten()
    valid     = np.isfinite(X).all(axis=1) & (mask_flat != 99)
    return X[valid], (mask_flat[valid] > 0).astype(np.uint8)


def load_dataset(s1_dir, mask_dir, tile_ids):
    """Assemble all pixels from a list of tile_ids into one training array."""
    X_list, y_list = [], []
    for tid in tqdm(tile_ids, desc='Loading tiles'):
        sp = os.path.join(s1_dir, tid)
        mp = os.path.join(mask_dir, tid)
        if not os.path.exists(sp) or not os.path.exists(mp):
            continue
        Xt, yt = load_tile_pixels(sp, mp)
        if len(Xt) == 0:
            continue
        X_list.append(Xt)
        y_list.append(yt)
    if not X_list:
        raise ValueError('No tiles could be loaded. Check paths and tile_ids.')
    return np.concatenate(X_list), np.concatenate(y_list)


def evaluate(model, s1_files, mask_lookup, threshold=0.5):
    """
    Pixel-level evaluation. Returns dict with all metrics.
    Iterates over s1_files, skips tiles without a matching mask.
    """
    all_yt, all_yp = [], []
    for s1_path in tqdm(s1_files, desc='Evaluating', leave=False):
        fname = os.path.basename(s1_path)
        mp    = mask_lookup.get(fname)
        if mp is None:
            continue
        with rasterio.open(s1_path) as src:
            img = src.read()
        with rasterio.open(mp) as src:
            mask = src.read(1)
        X     = np.column_stack([img[0].flatten(), img[1].flatten()])
        m     = mask.flatten()
        valid = np.isfinite(X).all(axis=1) & (m != 99)
        if valid.sum() == 0:
            continue
        proba = model.predict_proba(X[valid])[:, 1]
        yp    = (proba >= threshold).astype(np.uint8)
        yt    = (m[valid] > 0).astype(np.uint8)
        all_yt.append(yt)
        all_yp.append(yp)

    yt_all = np.concatenate(all_yt)
    yp_all = np.concatenate(all_yp)
    cm     = confusion_matrix(yt_all, yp_all, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    acc  = (tp + tn) / (tp + tn + fp + fn)
    pw   = tp / (tp + fp)  if (tp + fp)  else 0.0
    rw   = tp / (tp + fn)  if (tp + fn)  else 0.0
    fw   = 2*pw*rw / (pw+rw) if (pw+rw) else 0.0
    pnw  = tn / (tn + fn)  if (tn + fn)  else 0.0
    rnw  = tn / (tn + fp)  if (tn + fp)  else 0.0
    fnw  = 2*pnw*rnw / (pnw+rnw) if (pnw+rnw) else 0.0

    return {
        'water_precision': round(pw,  6), 'water_recall': round(rw,  6),
        'water_f1':        round(fw,  6),
        'nw_precision':    round(pnw, 6), 'nw_recall': round(rnw, 6),
        'nw_f1':           round(fnw, 6),
        'macro_f1':        round((fw + fnw) / 2, 6),
        'accuracy':        round(acc, 6),
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
    }


def append_csv(csv_path, row):
    """Append one row dict to a CSV, creating header on first write."""
    file_exists = os.path.exists(csv_path)
    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def select_tiles(meta, pct_int, rng):
    """
    Select pct_int% of each event's tiles (min 1 per event).
    pct_int is an integer percentage, e.g. 10 means 10%.
    Tiles within each event are shuffled before selection.
    Returns list of tile_id strings.
    """
    selected = []
    for ems, group in meta.groupby('ems_code'):
        tiles = group['tile_id'].tolist()
        rng.shuffle(tiles)
        n = max(1, round(len(tiles) * pct_int / 100))
        selected.extend(tiles[:n])
    rng.shuffle(selected)
    return selected


print('Helper functions defined.')


## Cell 5: Event split (80/20)

Split the STURM-Flood EMS events into train and validation sets.

- Split is on `ems_code` level, so the same event never appears in both sets.
- Controlled by `split_seed` for full reproducibility.
- This split is fixed for all subsequent pct and depth tests.
- The external test events come from a separate EMSR flood activation and must not overlap with the STURM-Flood training events. As long as the test activation is absent from the STURM-Flood metadata, no explicit exclusion from the split is needed; otherwise its events must be removed to avoid leakage.


In [ ]:
import numpy as np
import pandas as pd

# ── Load metadata ─────────────────────────────────────────────────────────────
metadata       = pd.read_csv(metadata_path, sep=';', quotechar='"')
available_s1   = set(os.listdir(sturm_s1_dir))
available_mask = set(os.listdir(sturm_mask_dir))
metadata       = metadata[metadata['tile_id'].isin(available_s1 & available_mask)].copy()

# ── 80/20 event split ─────────────────────────────────────────────────────────
all_events = sorted(metadata['ems_code'].unique())
n_train    = round(len(all_events) * train_ratio)

rng_split    = np.random.RandomState(split_seed)
shuffled     = rng_split.permutation(all_events).tolist()
train_events = sorted(shuffled[:n_train])
val_events   = sorted(shuffled[n_train:])

meta_train = metadata[metadata['ems_code'].isin(train_events)].copy()
meta_val   = metadata[metadata['ems_code'].isin(val_events)].copy()

print(f'Total events: {len(all_events)}  |  split_seed: {split_seed}')
print(f'Train events ({len(train_events)}): {train_events}')
print(f'Val   events ({len(val_events)}):  {val_events}')
print(f'\nTrain tiles available: {len(meta_train)}')
print(f'Val   tiles available: {len(meta_val)}')

print(f'\nEstimated tile counts:')
train_counts = meta_train.groupby('ems_code')['tile_id'].count()
val_counts   = meta_val.groupby('ems_code')['tile_id'].count()
for pct in train_pct_list:
    n_tr = sum(max(1, round(c * pct / 100)) for c in train_counts)
    print(f'  train_pct={pct}%  ->  ~{n_tr} train tiles')
n_val = sum(max(1, round(c * val_pct / 100)) for c in val_counts)
print(f'  val_pct={val_pct}%    ->  ~{n_val} val tiles')


## Cell 6: Pct test (learning curve)

Train RF at each `train_pct_list` value. Evaluate on validation split.
`max_depth` is fixed at `fixed_depth` during this step.

**Goal:** Find where performance plateaus or degrades (domain mismatch).
**After running:** Update `best_train_pct` in Cell 1.


In [ ]:
pct_csv = os.path.join(dirs['tables'], 'rf_pct_results.csv')

# ── Build validation tile set once ────────────────────────────────────────────
rng_val   = np.random.RandomState(tile_seed)
val_tiles = select_tiles(meta_val, val_pct, rng_val)

val_s1_files   = [os.path.join(sturm_s1_dir, t) for t in val_tiles
                  if os.path.exists(os.path.join(sturm_s1_dir, t))]
val_mask_lookup = {
    os.path.basename(f): os.path.join(sturm_mask_dir, os.path.basename(f))
    for f in val_s1_files
    if os.path.exists(os.path.join(sturm_mask_dir, os.path.basename(f)))
}

print('=' * 64)
print('  PCT TEST | LEARNING CURVE')
print(f'  max_depth: {fixed_depth} (fixed) | rf_seed: {rf_seed}')
print(f'  train_pct_list: {train_pct_list}  |  val_pct: {val_pct}%')
print(f'  Train events: {len(train_events)}  |  Val events: {len(val_events)}')
print(f'  Val tiles loaded: {len(val_s1_files)}')
print('=' * 64)

for train_pct in train_pct_list:
    rng_train = np.random.RandomState(tile_seed)
    tr_tiles  = select_tiles(meta_train, train_pct, rng_train)

    print(f"\n{'─'*50}")
    print(f'  train_pct={train_pct}%  ->  {len(tr_tiles)} train tiles')
    print(f"{'─'*50}")

    # Load training pixels
    t0 = time.time()
    X_tr, y_tr = load_dataset(sturm_s1_dir, sturm_mask_dir, tr_tiles)
    load_time  = time.time() - t0
    water_pct_tr = np.sum(y_tr == 1) / len(y_tr) * 100
    print(f'  {len(X_tr):,} train px | {water_pct_tr:.1f}% water | loaded {load_time:.0f}s')

    # Load val pixels
    t0 = time.time()
    X_val, y_val = load_dataset(sturm_s1_dir, sturm_mask_dir,
                                [os.path.basename(f) for f in val_s1_files])
    val_load_time = time.time() - t0
    water_pct_val = np.sum(y_val == 1) / len(y_val) * 100
    print(f'  {len(X_val):,} val px   | {water_pct_val:.1f}% water | loaded {val_load_time:.0f}s')

    # Train
    t0 = time.time()
    rf = RandomForestClassifier(
        n_estimators=rf_n_estimators, max_depth=fixed_depth,
        max_features=rf_max_features, random_state=rf_seed,
        n_jobs=rf_n_jobs, class_weight=rf_class_weight
    )
    rf.fit(X_tr, y_tr)
    train_time = time.time() - t0
    imp = rf.feature_importances_
    print(f'  Train: {train_time:.1f}s | VV={imp[0]:.4f} VH={imp[1]:.4f}')

    # Evaluate on val split
    t0 = time.time()
    metrics = evaluate(rf, val_s1_files, val_mask_lookup, threshold=score_threshold)
    eval_time = time.time() - t0
    print(f'  [VAL] Acc={metrics["accuracy"]:.4f}  Macro F1={metrics["macro_f1"]:.4f}  '
          f'Water P/R/F1={metrics["water_precision"]:.3f}/'
          f'{metrics["water_recall"]:.3f}/{metrics["water_f1"]:.3f}')

    row = {
        'timestamp':          datetime.now().isoformat(),
        'run_type':           'pct_test',
        'split_seed':         split_seed,
        'train_pct':          train_pct,
        'val_pct':            val_pct,
        'max_depth':          str(fixed_depth),
        'n_estimators':       rf_n_estimators,
        'class_weight':       str(rf_class_weight),
        'train_events':       len(train_events),
        'val_events':         len(val_events),
        'train_tiles':        len(tr_tiles),
        'val_tiles':          len(val_s1_files),
        'train_pixels_total': len(X_tr),
        'val_pixels_total':   len(X_val),
        'vv_importance':      round(float(imp[0]), 4),
        'vh_importance':      round(float(imp[1]), 4),
        'train_time_s':       round(train_time, 1),
        'eval_time_s':        round(eval_time, 1),
        **metrics
    }
    append_csv(pct_csv, row)
    print(f'  Appended to {pct_csv}')

    del rf, X_tr, y_tr, X_val, y_val
    gc.collect()

print(f'\nPct test done. Results: {pct_csv}')
print(f'Update best_train_pct in Cell 1 before running Cell 7.')


## Cell 6b: Plot pct test results


In [ ]:
pct_df = pd.read_csv(os.path.join(dirs['tables'], 'rf_pct_results.csv'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(pct_df['train_pct'], pct_df['water_f1'],  'o-', label='Water F1', lw=2)
ax.plot(pct_df['train_pct'], pct_df['macro_f1'],  's--', label='Macro F1', lw=2)
ax.plot(pct_df['train_pct'], pct_df['accuracy'],  '^:', label='Accuracy', lw=2)
ax.set_xlabel('train_pct (%)'); ax.set_ylabel('Score')
ax.set_title('Learning Curve (Validation Split)')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(pct_df['train_pct'], pct_df['water_precision'], 'o-', label='Precision', lw=2)
ax.plot(pct_df['train_pct'], pct_df['water_recall'],    's-', label='Recall', lw=2)
ax.set_xlabel('train_pct (%)'); ax.set_ylabel('Score')
ax.set_title('Water: Precision vs Recall')
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle(f'RF Pct Test | split_seed={split_seed} | max_depth={fixed_depth}')
plt.tight_layout()
plt.savefig(os.path.join(dirs['tables'], 'rf_pct_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print(pct_df[['train_pct', 'water_f1', 'macro_f1', 'water_precision',
              'water_recall', 'accuracy']].to_string(index=False))


## Cell 7: Depth test (max_depth search)

Train RF at `best_train_pct` (set in Cell 1 after pct test), varying `max_depth`.
Training data is loaded once and reused for all depths.
Evaluation is on the same validation split as Cell 6.

**After running:** Update `best_max_depth` in Cell 1.


In [ ]:
depth_csv = os.path.join(dirs['tables'], 'rf_depth_results.csv')

# ── Load training data once ───────────────────────────────────────────────────
rng_train_d = np.random.RandomState(tile_seed)
tr_tiles_d  = select_tiles(meta_train, best_train_pct, rng_train_d)

t0 = time.time()
X_tr_d, y_tr_d = load_dataset(sturm_s1_dir, sturm_mask_dir, tr_tiles_d)
load_time = time.time() - t0
water_pct_tr = np.sum(y_tr_d == 1) / len(y_tr_d) * 100

# ── Load val data once ────────────────────────────────────────────────────────
t0 = time.time()
X_val_d, y_val_d = load_dataset(sturm_s1_dir, sturm_mask_dir,
                                [os.path.basename(f) for f in val_s1_files])
val_load_time = time.time() - t0
water_pct_val = np.sum(y_val_d == 1) / len(y_val_d) * 100

print('=' * 64)
print('  DEPTH TEST | MAX_DEPTH SEARCH')
print(f'  train_pct={best_train_pct}%  ->  {len(tr_tiles_d)} train tiles')
print(f'  {len(X_tr_d):,} train px | {water_pct_tr:.1f}% water | loaded {load_time:.0f}s')
print(f'  {len(X_val_d):,} val px   | {water_pct_val:.1f}% water | loaded {val_load_time:.0f}s')
print(f'  Depths to test: {depth_list}')
print('=' * 64)

for depth in depth_list:
    print(f"\n{'─'*50}")
    print(f'  max_depth: {depth}')

    t0 = time.time()
    rf = RandomForestClassifier(
        n_estimators=rf_n_estimators, max_depth=depth,
        max_features=rf_max_features, random_state=rf_seed,
        n_jobs=rf_n_jobs, class_weight=rf_class_weight
    )
    rf.fit(X_tr_d, y_tr_d)
    train_time = time.time() - t0
    imp = rf.feature_importances_
    print(f'  Train: {train_time:.1f}s | VV={imp[0]:.4f} VH={imp[1]:.4f}')

    t0 = time.time()
    metrics = evaluate(rf, val_s1_files, val_mask_lookup, threshold=score_threshold)
    eval_time = time.time() - t0
    print(f'  [VAL] Acc={metrics["accuracy"]:.4f}  Macro F1={metrics["macro_f1"]:.4f}  '
          f'Water P/R/F1={metrics["water_precision"]:.3f}/'
          f'{metrics["water_recall"]:.3f}/{metrics["water_f1"]:.3f}')

    row = {
        'timestamp':          datetime.now().isoformat(),
        'run_type':           'depth_test',
        'split_seed':         split_seed,
        'train_pct':          best_train_pct,
        'val_pct':            val_pct,
        'max_depth':          str(depth),
        'n_estimators':       rf_n_estimators,
        'class_weight':       str(rf_class_weight),
        'train_events':       len(train_events),
        'val_events':         len(val_events),
        'train_tiles':        len(tr_tiles_d),
        'val_tiles':          len(val_s1_files),
        'train_pixels_total': len(X_tr_d),
        'val_pixels_total':   len(X_val_d),
        'vv_importance':      round(float(imp[0]), 4),
        'vh_importance':      round(float(imp[1]), 4),
        'train_time_s':       round(train_time, 1),
        'eval_time_s':        round(eval_time, 1),
        **metrics
    }
    append_csv(depth_csv, row)
    print(f'  Appended to {depth_csv}')

    del rf
    gc.collect()

del X_tr_d, y_tr_d, X_val_d, y_val_d
gc.collect()
print(f'\nDepth test done. Results: {depth_csv}')
print(f'Update best_max_depth in Cell 1 before running Cell 8.')


## Cell 7b: Plot depth test results


In [ ]:
depth_df = pd.read_csv(os.path.join(dirs['tables'], 'rf_depth_results.csv'))

fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(depth_df))
ax.plot(x, depth_df['water_f1'],  'o-', label='Water F1', lw=2)
ax.plot(x, depth_df['macro_f1'],  's--', label='Macro F1', lw=2)
ax.plot(x, depth_df['accuracy'],  '^:', label='Accuracy', lw=2)
ax.set_xticks(list(x))
ax.set_xticklabels(depth_df['max_depth'].astype(str))
ax.set_xlabel('max_depth'); ax.set_ylabel('Score')
ax.set_title(f'Depth Test (Validation Split) | train_pct={best_train_pct}%')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(dirs['tables'], 'rf_depth_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print(depth_df[['max_depth', 'water_f1', 'macro_f1', 'water_precision',
                'water_recall', 'accuracy']].to_string(index=False))


## Cell 8: Final model (train and save)

Train the final RF model with `best_train_pct` and `best_max_depth` (set in Cell 1).
Saves model to `models/` and config to `configs/`.

The saved model is used in Cell 9 for external testing.


In [ ]:
from datetime import datetime as dt

model_id   = f"rf_{best_train_pct}pct_depth{best_max_depth}_seed{tile_seed}_{dt.now().strftime('%Y%m%d_%H%M')}"
model_path = os.path.join(dirs['models'], f'{model_id}.joblib')
config_path = os.path.join(dirs['configs'], f'{model_id}.json')

# Train on full best_train_pct selection
rng_final  = np.random.RandomState(tile_seed)
tr_final   = select_tiles(meta_train, best_train_pct, rng_final)

print('=' * 64)
print('  FINAL MODEL TRAINING')
print(f'  train_pct={best_train_pct}%  ->  {len(tr_final)} train tiles')
print(f'  max_depth={best_max_depth} | n_estimators={rf_n_estimators}')
print('=' * 64)

t0 = time.time()
X_f, y_f = load_dataset(sturm_s1_dir, sturm_mask_dir, tr_final)
load_time = time.time() - t0
water_pct_f = np.sum(y_f == 1) / len(y_f) * 100
print(f'{len(X_f):,} px | {water_pct_f:.1f}% water | loaded {load_time:.0f}s')

t0 = time.time()
final_rf = RandomForestClassifier(
    n_estimators=rf_n_estimators, max_depth=best_max_depth,
    max_features=rf_max_features, random_state=rf_seed,
    n_jobs=rf_n_jobs, class_weight=rf_class_weight
)
final_rf.fit(X_f, y_f)
train_time = time.time() - t0
imp = final_rf.feature_importances_
print(f'Train: {train_time:.1f}s | VV={imp[0]:.4f} VH={imp[1]:.4f}')

# Save model
joblib.dump(final_rf, model_path)
print(f'\nModel saved: {model_path}')

# Save config
config = {
    'model_id':         model_id,
    'split_seed':       split_seed,
    'train_pct':        best_train_pct,
    'val_pct':          val_pct,
    'max_depth':        str(best_max_depth),
    'n_estimators':     rf_n_estimators,
    'max_features':     rf_max_features,
    'class_weight':     str(rf_class_weight),
    'score_threshold':  score_threshold,
    'tile_seed':        tile_seed,
    'rf_seed':          rf_seed,
    'train_events':     train_events,
    'val_events':       val_events,
    'train_tiles':      len(tr_final),
    'train_pixels':     len(X_f),
    'train_water_pct':  round(water_pct_f, 2),
    'vv_importance':    round(float(imp[0]), 4),
    'vh_importance':    round(float(imp[1]), 4),
}
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'Config saved: {config_path}')

del X_f, y_f
gc.collect()
print(f'\nfinal_rf ready for Cell 9.')


## Cell 9: External test (all EMSR areas)

Run the final model on all three external test areas. Saves metrics to
`rf_external_test_results.csv` and GeoTIFF predictions to `geotiff_predictions/`.

GeoTIFF encoding: 0 = non-water, 1 = water, 255 = nodata. Files named
`RF_<tile_filename>` for TileExplorer compatibility.

### Requirements

This cell depends on variables defined in Cells 1–8 (paths, helper functions,
`emsr_areas`, `final_rf`, `model_id`, etc.). Run the full pipeline before
executing this cell.

### When to use Cell 9 vs Cell 10

- **Cell 9**: first evaluation after training. Runs all three areas in
  one pass and produces the initial result rows in
  `rf_external_test_results.csv`.
- **Cell 10**: re-evaluation of a single area without rerunning the
  pipeline. Self-contained; loads the saved model from disk. Use after
  ground truth updates or when only one area needs to be reprocessed.


In [ ]:
ext_csv = os.path.join(dirs['tables'], 'rf_external_test_results.csv')

# Load model if not already in memory
if 'final_rf' not in dir() or final_rf is None:
    final_rf    = joblib.load(model_path)
    config_data = json.load(open(config_path))
    print(f'Model loaded from {model_path}')

print('=' * 64)
print('  EXTERNAL TEST | ALL EMSR AREAS')
print(f'  Model: {model_id}')
print('=' * 64)

ext_results = []

for area, cfg in emsr_areas.items():
    print(f"\n{'─'*50}")
    print(f'  Area: {area.upper()}  ({len(cfg["s1_files"])} tiles)')
    print(f"{'─'*50}")

    # ── Evaluate ─────────────────────────────────────────────────────────────
    t0 = time.time()
    metrics = evaluate(final_rf, cfg['s1_files'], cfg['mask_lookup'],
                       threshold=score_threshold)
    eval_time = time.time() - t0
    print(f'  Acc={metrics["accuracy"]:.4f}  Macro F1={metrics["macro_f1"]:.4f}  '
          f'Water P/R/F1={metrics["water_precision"]:.3f}/'
          f'{metrics["water_recall"]:.3f}/{metrics["water_f1"]:.3f}  ({eval_time:.0f}s)')

    row = {
        'timestamp':    datetime.now().isoformat(),
        'model_id':     model_id,
        'run_type':     'external_test',
        'split_seed':   split_seed,
        'train_pct':    best_train_pct,
        'val_pct':      val_pct,
        'max_depth':    str(best_max_depth),
        'n_estimators': rf_n_estimators,
        'class_weight': str(rf_class_weight),
        'test_area':    area,
        'n_test_tiles': len(cfg['s1_files']),
        'eval_time_s':  round(eval_time, 1),
        'model_path':   model_path,
        'config_path':  config_path,
        **metrics
    }
    append_csv(ext_csv, row)
    ext_results.append(row)

    # ── Save GeoTIFF predictions ──────────────────────────────────────────────
    print(f'  Saving GeoTIFF predictions...')
    saved = 0
    for s1_path in tqdm(cfg['s1_files'], desc=f'  {area}'):
        fname = os.path.basename(s1_path)
        mp    = cfg['mask_lookup'].get(fname)
        if mp is None:
            continue
        with rasterio.open(s1_path) as src:
            img     = src.read()
            profile = src.profile.copy()
        mask_arr = rasterio.open(mp).read(1)
        X    = np.column_stack([img[0].flatten(), img[1].flatten()])
        m    = mask_arr.flatten()
        valid = np.isfinite(X).all(axis=1) & (m != 99)
        pred = np.full(len(m), 255, dtype=np.uint8)
        if valid.sum() > 0:
            proba       = final_rf.predict_proba(X[valid])[:, 1]
            pred[valid] = (proba >= score_threshold).astype(np.uint8)
        profile.update(count=1, dtype='uint8', nodata=255)
        out_path = os.path.join(dirs['geotiff'], f'RF_{fname}')
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(pred.reshape(128, 128)[np.newaxis, :, :])
        saved += 1
    print(f'  Saved {saved} GeoTIFF predictions')

# ── Summary ───────────────────────────────────────────────────────────────────
ext_df = pd.DataFrame(ext_results)
print('\n' + '=' * 64)
print('  EXTERNAL TEST SUMMARY')
print('=' * 64)
print(ext_df[['test_area', 'water_f1', 'macro_f1', 'water_precision',
              'water_recall', 'accuracy', 'n_test_tiles']].to_string(index=False))
print(f'\nMetrics saved to: {ext_csv}')
print(f'GeoTIFFs saved to: {dirs["geotiff"]}')


## Cell 10: Re-evaluate single area (standalone)

Standalone cell for re-running RF inference on a single test area
without re-running the entire training pipeline (Cells 1–9).

### When to use

- Ground truth has been updated (e.g. corrected `CLASS_MAPPING` in the
  data preparation script) and metrics need to be recomputed against the
  new masks.
- A specific area needs to be re-evaluated in isolation, without rerunning
  training or evaluating the other test areas.
- The Colab session has been restarted and only inference is needed.

### Requirements

- A trained model: `models/<model_id>.joblib`
- Matching config: `configs/<model_id>.json`
- S1 tiles and corresponding flood masks for the target area

### What the cell does

1. Mounts Google Drive (no remount if already mounted).
2. Loads the trained Random Forest model and its config from Drive.
3. Reads S1 + mask tiles for the selected area.
4. Predicts water/non-water per pixel and computes metrics
   (accuracy, macro F1, water precision/recall/F1).
5. Appends a single result row to `tables/rf_external_test_results.csv`
   with `run_type = 'external_test_rerun'`.
6. Writes binary prediction GeoTIFFs to `geotiff_predictions/`,
   overwriting any existing files with the same name.

### Configuration

Edit the four variables under **CONFIG** before running:

- `output_base`: path to the RF results folder.
- `model_id`: filename of the model (without the `.joblib` extension).
- `test_area`: short name used in the CSV (e.g. `'goteborg'`).
- `area_s1_dir`, `area_mask_dir`: paths to the area's S1 and mask tiles.

The cell is fully self-contained and can be run from a fresh Colab
session, with no need to execute any of the upstream cells first.

In [ ]:
# ====================================================================
# Cell 10: Re-evaluate single area (standalone)
# ====================================================================
# Loads a trained RF model and re-evaluates it against a single
# external test area. Self-contained, does not depend on variables
# from earlier cells.
#
# Writes a CSV row with the same column structure as Cell 9, so rows
# from both cells stack consistently in rf_external_test_results.csv.
# ====================================================================

import os
import json
import glob
import time
from datetime import datetime

import numpy as np
import pandas as pd
import rasterio
import joblib
from tqdm import tqdm
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix)

from google.colab import drive
drive.mount('/content/drive', force_remount=True)


# ─── CONFIG ─────────────────────────────────────────────────────────
# Path to the RF results folder (must contain models/, configs/,
# tables/ and geotiff_predictions/ subfolders).
output_base = '/content/drive/MyDrive/Examensarbete/RF_results_v16'

# Model to evaluate. Filename of the .joblib without the extension.
# Inspect output_base/models/ to find the exact name.
model_id = 'rf_5pct_depth3_seed42_20260416_1100'

# Target area
test_area     = 'goteborg'
area_s1_dir   = '/content/drive/MyDrive/Examensarbete/Dataset_goteborg_final/Sentinel1/S1'
area_mask_dir = '/content/drive/MyDrive/Examensarbete/Dataset_goteborg_final/Sentinel1/Floodmaps'

# Whether to also save GeoTIFF predictions (set False to skip if model
# input is unchanged and predictions are already on disk).
save_geotiff = True


# ─── LOAD MODEL + CONFIG ────────────────────────────────────────────
model_path  = os.path.join(output_base, 'models',  f'{model_id}.joblib')
config_path = os.path.join(output_base, 'configs', f'{model_id}.json')

assert os.path.exists(model_path),  f'Model not found: {model_path}'
assert os.path.exists(config_path), f'Config not found: {config_path}'

rf       = joblib.load(model_path)
cfg_data = json.load(open(config_path))
score_threshold = cfg_data.get('score_threshold', 0.5)

print(f'✓ Loaded model: {model_id}')
print(f'  pct={cfg_data.get("train_pct")}  '
      f'depth={cfg_data.get("max_depth")}  '
      f'seed={cfg_data.get("rf_seed")}  '
      f'threshold={score_threshold}')


# ─── BUILD FILE LIST + MASK LOOKUP ──────────────────────────────────
s1_files    = sorted(glob.glob(os.path.join(area_s1_dir, '*.tif')))
mask_files  = glob.glob(os.path.join(area_mask_dir, '*.tif'))
mask_lookup = {os.path.basename(p): p for p in mask_files}

print(f'\n  Area: {test_area.upper()}')
print(f'  S1 tiles: {len(s1_files)}   masks: {len(mask_files)}')


# ─── EVALUATE ───────────────────────────────────────────────────────
print(f'\n{"─" * 50}\n  Evaluating...\n{"─" * 50}')

all_y_true, all_y_pred = [], []
t0 = time.time()

for s1_path in tqdm(s1_files, desc=test_area):
    fname = os.path.basename(s1_path)
    mp    = mask_lookup.get(fname)
    if mp is None:
        continue

    with rasterio.open(s1_path) as src:
        img = src.read()
    mask_arr = rasterio.open(mp).read(1)

    X     = np.column_stack([img[0].flatten(), img[1].flatten()])
    m     = mask_arr.flatten()
    valid = np.isfinite(X).all(axis=1) & (m != 99)
    if valid.sum() == 0:
        continue

    proba  = rf.predict_proba(X[valid])[:, 1]
    y_pred = (proba >= score_threshold).astype(np.uint8)
    y_true = (m[valid] > 0).astype(np.uint8)

    all_y_true.append(y_true)
    all_y_pred.append(y_pred)

y_true    = np.concatenate(all_y_true)
y_pred    = np.concatenate(all_y_pred)
eval_time = time.time() - t0

# Compute full set of metrics matching Cell 9 schema
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

metrics = {
    'water_precision': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
    'water_recall':    recall_score(y_true, y_pred, pos_label=1, zero_division=0),
    'water_f1':        f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    'nw_precision':    precision_score(y_true, y_pred, pos_label=0, zero_division=0),
    'nw_recall':       recall_score(y_true, y_pred, pos_label=0, zero_division=0),
    'nw_f1':           f1_score(y_true, y_pred, pos_label=0, zero_division=0),
    'macro_f1':        f1_score(y_true, y_pred, average='macro'),
    'accuracy':        accuracy_score(y_true, y_pred),
    'tp':              int(tp),
    'tn':              int(tn),
    'fp':              int(fp),
    'fn':              int(fn),
}

print(f'\n  Acc={metrics["accuracy"]:.4f}  '
      f'Macro F1={metrics["macro_f1"]:.4f}  '
      f'Water P/R/F1={metrics["water_precision"]:.3f}/'
      f'{metrics["water_recall"]:.3f}/{metrics["water_f1"]:.3f}  '
      f'({eval_time:.0f}s)')


# ─── APPEND CSV ROW ─────────────────────────────────────────────────
# Schema must match Cell 9 exactly so rows align in the same CSV.
ext_csv = os.path.join(output_base, 'tables', 'rf_external_test_results.csv')

row = {
    'timestamp':       datetime.now().isoformat(),
    'model_id':        model_id,
    'run_type':        'external_test_rerun',
    'split_seed':      cfg_data.get('split_seed', 42),
    'train_pct':       cfg_data.get('train_pct'),
    'val_pct':         cfg_data.get('val_pct'),
    'max_depth':       cfg_data.get('max_depth'),
    'n_estimators':    cfg_data.get('n_estimators', 100),
    'class_weight':    cfg_data.get('class_weight', 'balanced'),
    'test_area':       test_area,
    'n_test_tiles':    len(s1_files),
    'eval_time_s':     round(eval_time, 1),
    'model_path':      model_path,
    'config_path':     config_path,
    'water_precision': metrics['water_precision'],
    'water_recall':    metrics['water_recall'],
    'water_f1':        metrics['water_f1'],
    'nw_precision':    metrics['nw_precision'],
    'nw_recall':       metrics['nw_recall'],
    'nw_f1':           metrics['nw_f1'],
    'macro_f1':        metrics['macro_f1'],
    'accuracy':        metrics['accuracy'],
    'tp':              metrics['tp'],
    'tn':              metrics['tn'],
    'fp':              metrics['fp'],
    'fn':              metrics['fn'],
}

# Read existing CSV header to ensure column order matches
if os.path.exists(ext_csv):
    existing_header = pd.read_csv(ext_csv, nrows=0).columns.tolist()
    # Build row in the order of existing header (missing fields become empty)
    df = pd.DataFrame([{col: row.get(col, '') for col in existing_header}])
    df.to_csv(ext_csv, mode='a', header=False, index=False)
else:
    df = pd.DataFrame([row])
    df.to_csv(ext_csv, index=False)

print(f'\n  ✓ CSV updated: {ext_csv}')


# ─── SAVE GEOTIFF PREDICTIONS ───────────────────────────────────────
if save_geotiff:
    geotiff_dir = os.path.join(output_base, 'geotiff_predictions')
    os.makedirs(geotiff_dir, exist_ok=True)

    print(f'\n  Saving GeoTIFF predictions...')
    saved = 0

    for s1_path in tqdm(s1_files, desc=f'  {test_area}'):
        fname = os.path.basename(s1_path)
        mp    = mask_lookup.get(fname)
        if mp is None:
            continue

        with rasterio.open(s1_path) as src:
            img     = src.read()
            profile = src.profile.copy()
        mask_arr = rasterio.open(mp).read(1)

        X     = np.column_stack([img[0].flatten(), img[1].flatten()])
        m     = mask_arr.flatten()
        valid = np.isfinite(X).all(axis=1) & (m != 99)

        pred = np.full(len(m), 255, dtype=np.uint8)
        if valid.sum() > 0:
            proba       = rf.predict_proba(X[valid])[:, 1]
            pred[valid] = (proba >= score_threshold).astype(np.uint8)

        profile.update(count=1, dtype='uint8', nodata=255)
        out_path = os.path.join(geotiff_dir, f'RF_{fname}')
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(pred.reshape(128, 128)[np.newaxis, :, :])
        saved += 1

    print(f'  ✓ Saved {saved} GeoTIFF predictions')
else:
    print('\n  Skipping GeoTIFF save (save_geotiff=False)')